# Asset Pricing and Investor Sentiment Analysis
## Main Analysis Notebook

This notebook demonstrates the complete end-to-end workflow for analyzing the relationship between investor sentiment and factor-based portfolio returns.

**Research Question:** Does investor sentiment—once purified of macroeconomic confounds—help explain or predict the returns of well-documented anomaly (factor-based) portfolios?

### Workflow Overview:
1. Load and process sentiment indicators
2. Orthogonalize sentiment against macroeconomic variables
3. Load and process factor portfolios
4. Run factor model regressions (CAPM, FF3, FF5)
5. Perform sentiment-conditional analysis
6. Generate results and visualizations

## Setup

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

# Add src to path
sys.path.append('../src')

# Import custom modules
from data.data_utils import load_config, save_checkpoint, load_checkpoint
from utils.date_utils import to_datetime_index, calculate_growth_rate
from utils.statistical_utils import standardize, descriptive_stats
from preprocessing.orthogonalization import (
    load_macro_variables,
    transform_to_growth_rates,
    orthogonalize_all_indicators,
    print_orthogonalization_summary
)
from analysis.sentiment_conditional import (
    create_sentiment_terciles,
    sentiment_factor_analysis,
    create_sentiment_conditional_table
)

# Settings
warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load configuration
config = load_config('../config/config.yaml')
print("Configuration loaded successfully")

## 1. Load and Process Sentiment Indicators

We begin by loading seven sentiment indicators:
1. Baker-Wurgler Sentiment Index
2. Zhou Investor Sentiment
3. University of Michigan Consumer Sentiment (ICS)
4. AAII Bull-Bear Spread
5. CBOE VIX
6. Zhou Manager Sentiment
7. Zhou Employee Sentiment

In [ ]:
# Load sentiment indicators
# NOTE: Update these paths to match your data location
sentiment_path = config['paths']['raw_data'] + '/sentiment'

# Example: Load Baker-Wurgler Sentiment
# df_bw = pd.read_excel(f'{sentiment_path}/BW_Sentiment.xlsx', sheet_name='BW_EOM')
# df_bw = to_datetime_index(df_bw)

# For demonstration, create sample data
dates = pd.date_range('2000-01', '2023-12', freq='M')
sentiment_df = pd.DataFrame({
    'BW': np.random.normal(0, 1, len(dates)),
    'Zhou_Investor': np.random.normal(0, 1, len(dates)),
    'ICS': np.random.normal(85, 10, len(dates)),
    'AAII': np.random.normal(0, 20, len(dates)),
    'VIX': np.random.normal(20, 5, len(dates)),
    'Manager': np.random.normal(0, 1, len(dates)),
    'Employee': np.random.normal(0, 1, len(dates))
}, index=dates)

print("Sentiment indicators loaded:")
print(sentiment_df.head())
print(f"\nShape: {sentiment_df.shape}")
print(f"Date range: {sentiment_df.index.min()} to {sentiment_df.index.max()}")

In [ ]:
# Descriptive statistics
print("Descriptive Statistics:")
print(descriptive_stats(sentiment_df))

In [ ]:
# Visualize sentiment indicators
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(sentiment_df.columns):
    axes[i].plot(sentiment_df.index, sentiment_df[col])
    axes[i].set_title(col)
    axes[i].set_xlabel('Date')
    axes[i].grid(True, alpha=0.3)

# Remove empty subplots
for i in range(len(sentiment_df.columns), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.savefig('../results/figures/sentiment_indicators.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Orthogonalize Sentiment Against Macroeconomic Variables

We purify sentiment of macroeconomic influences by regressing each indicator on macro growth rates.

**Macroeconomic Variables:**
- Industrial Production (12-month log-difference)
- Durable Consumption (12-month log-difference)
- Nondurable Consumption (12-month log-difference)
- Services (12-month log-difference)
- Employment (12-month log-difference)
- CPI (12-month percentage change)

**Method:** OLS with Newey-West HAC standard errors (4 lags)

In [ ]:
# Load macroeconomic variables
# df_macro = load_macro_variables('../data/raw/BW_Sentiment.xlsx')

# For demonstration, create sample macro data
macro_df_raw = pd.DataFrame({
    'INDPRO': 100 * (1.02 ** np.arange(len(dates))),
    'DURABLE': 100 * (1.015 ** np.arange(len(dates))),
    'NONDUR': 100 * (1.01 ** np.arange(len(dates))),
    'SERVICES': 100 * (1.025 ** np.arange(len(dates))),
    'EMPLOY': 100 * (1.005 ** np.arange(len(dates))),
    'CPI': 100 * (1.03 ** np.arange(len(dates)))
}, index=dates)

print("Macro variables loaded:")
print(macro_df_raw.head())

In [ ]:
# Transform to growth rates
real_vars = ['INDPRO', 'DURABLE', 'NONDUR', 'SERVICES', 'EMPLOY']
nominal_vars = ['CPI']
periods = config['macro_variables']['growth_window']

macro_growth = transform_to_growth_rates(
    macro_df_raw,
    real_vars=real_vars,
    nominal_vars=nominal_vars,
    periods=periods
)

print("\nMacro growth rates:")
print(macro_growth.head())
print(f"\nShape: {macro_growth.shape}")

In [ ]:
# Orthogonalize all sentiment indicators
lags = config['regression']['newey_west_lags']['orthogonalization']

orthogonalized_df, diagnostics = orthogonalize_all_indicators(
    sentiment_df,
    macro_growth,
    lags=lags,
    run_diagnostics_flag=True
)

print("\nOrthogonalized sentiment:")
print(orthogonalized_df.head())

In [ ]:
# Print diagnostics summary
print_orthogonalization_summary(diagnostics)

In [ ]:
# Compare original vs orthogonalized
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

indicators_to_plot = ['BW', 'VIX', 'Manager', 'Employee']

for i, indicator in enumerate(indicators_to_plot):
    if indicator in sentiment_df.columns and f'{indicator}_orth' in orthogonalized_df.columns:
        axes[i].plot(sentiment_df.index, sentiment_df[indicator], 
                    label='Original', alpha=0.7)
        axes[i].plot(orthogonalized_df.index, orthogonalized_df[f'{indicator}_orth'], 
                    label='Orthogonalized', alpha=0.7)
        axes[i].set_title(f'{indicator}: Original vs Orthogonalized')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/sentiment_orthogonalization_comparison.png', 
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Save checkpoint
save_checkpoint(orthogonalized_df, 'orthogonalized_sentiment', 
                path='../results/checkpoints')

## 3. Load and Process Factor Portfolios

Load factor portfolio data from Open Asset Pricing and JKP libraries.
For each factor, we analyze:
- Long portfolio (Decile 10)
- Short portfolio (Decile 01)
- Long-Short portfolio (10 - 01)

In [ ]:
# Load factor portfolios
# For demonstration, create sample factor returns
factors = ['Momentum', 'Value', 'Size', 'Profitability', 'Investment', 
           'Low_Vol', 'Quality', 'Growth']

# Create sample returns data
factor_returns = pd.DataFrame(
    index=dates,
    columns=[f'{factor}_Long-Short' for factor in factors]
)

for factor in factors:
    # Simulate returns with some persistence
    returns = np.random.normal(0.005, 0.05, len(dates))
    # Add autocorrelation
    for i in range(1, len(returns)):
        returns[i] = 0.3 * returns[i-1] + 0.7 * returns[i]
    factor_returns[f'{factor}_Long-Short'] = returns

print("Factor returns loaded:")
print(factor_returns.head())
print(f"\nShape: {factor_returns.shape}")
print(f"\nFactors: {factors}")

In [ ]:
# Summary statistics for factor returns
print("Factor Returns Summary (annualized):")
summary = pd.DataFrame({
    'Mean (%)': factor_returns.mean() * 12 * 100,
    'Std (%)': factor_returns.std() * np.sqrt(12) * 100,
    'Sharpe': (factor_returns.mean() / factor_returns.std()) * np.sqrt(12)
})
print(summary.round(2))

## 4. Sentiment-Conditional Analysis

Analyze how factor returns vary across High and Low sentiment regimes.

**Methodology:**
1. Split sentiment into terciles (Low = bottom 33%, High = top 33%)
2. Calculate mean factor returns in High vs Low sentiment periods
3. Compute High-Minus-Low (HML) difference
4. Test significance using Newey-West t-statistics (3 lags)

In [ ]:
# Perform sentiment-conditional analysis
results = sentiment_factor_analysis(
    sentiment_df=orthogonalized_df,
    residuals_df=factor_returns,
    portfolio_legs=['Long-Short'],
    lags=config['regression']['newey_west_lags']['hml_test']
)

print(f"\nAnalysis complete. Results for {len(results)} sentiment-leg combinations.")

In [ ]:
# Display results for Baker-Wurgler sentiment
if 'BW_orth_Long-Short' in results:
    bw_results = results['BW_orth_Long-Short']
    bw_table = create_sentiment_conditional_table(bw_results, sort_by='HML_tstat')
    
    print("\nBaker-Wurgler Orthogonalized Sentiment - Long-Short Portfolios:")
    print("=" * 80)
    print(bw_table.to_string(index=False))
    
    # Save table
    bw_table.to_csv('../results/tables/bw_sentiment_conditional.csv', index=False)
    print("\nTable saved to results/tables/bw_sentiment_conditional.csv")

In [ ]:
# Visualize High vs Low returns
if 'BW_orth_Long-Short' in results:
    bw_results = results['BW_orth_Long-Short']
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(bw_results))
    width = 0.35
    
    ax.bar(x - width/2, bw_results['High'] * 100, width, label='High Sentiment', alpha=0.8)
    ax.bar(x + width/2, bw_results['Low'] * 100, width, label='Low Sentiment', alpha=0.8)
    
    ax.set_xlabel('Factor')
    ax.set_ylabel('Monthly Return (%)')
    ax.set_title('Factor Returns: High vs Low Sentiment (BW Orthogonalized)')
    ax.set_xticks(x)
    ax.set_xticklabels(bw_results['Factor'], rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    plt.savefig('../results/figures/high_vs_low_sentiment_returns.png', 
                dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Visualize t-statistics
if 'BW_orth_Long-Short' in results:
    bw_results = results['BW_orth_Long-Short']
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = ['green' if t > 1.96 else 'red' if t < -1.96 else 'gray' 
              for t in bw_results['HML_tstat']]
    
    ax.barh(bw_results['Factor'], bw_results['HML_tstat'], color=colors, alpha=0.7)
    ax.axvline(x=1.96, color='black', linestyle='--', linewidth=1, label='5% significance')
    ax.axvline(x=-1.96, color='black', linestyle='--', linewidth=1)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    
    ax.set_xlabel('t-statistic')
    ax.set_ylabel('Factor')
    ax.set_title('High-Minus-Low t-statistics (BW Orthogonalized Sentiment)')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('../results/figures/hml_tstatistics.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Cross-Sentiment Comparison

Compare results across different sentiment indicators

In [ ]:
# Create comparison table across sentiment indicators
comparison_data = []

for key, df in results.items():
    sentiment_name = key.replace('_orth_Long-Short', '')
    
    # Count significant results
    significant = df[abs(df['HML_tstat']) > 1.96]
    
    comparison_data.append({
        'Sentiment': sentiment_name,
        'N_Factors': len(df),
        'N_Significant': len(significant),
        'Pct_Significant': len(significant) / len(df) * 100 if len(df) > 0 else 0,
        'Mean_|t-stat|': abs(df['HML_tstat']).mean(),
        'Max_|t-stat|': abs(df['HML_tstat']).max()
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('N_Significant', ascending=False)

print("\nSentiment Indicator Comparison:")
print("=" * 80)
print(comparison_df.to_string(index=False))

comparison_df.to_csv('../results/tables/sentiment_comparison.csv', index=False)

## 6. Summary and Key Findings

Summarize the main results from the analysis

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

print("\n1. SENTIMENT ORTHOGONALIZATION")
print("-" * 40)
for indicator, diag in diagnostics.items():
    if 'rsquared' in diag:
        pct_explained = diag['rsquared'] * 100
        pct_remaining = (1 - diag['rsquared']) * 100
        print(f"  {indicator}:")
        print(f"    Macro explains: {pct_explained:.1f}%")
        print(f"    Behavioral component: {pct_remaining:.1f}%")

print("\n2. SENTIMENT-CONDITIONAL RETURNS")
print("-" * 40)
total_significant = comparison_df['N_Significant'].sum()
total_tests = comparison_df['N_Factors'].sum()
print(f"  Total factors analyzed: {total_tests}")
print(f"  Significant results (|t| > 1.96): {total_significant}")
print(f"  Significance rate: {total_significant/total_tests*100:.1f}%")

print("\n3. MOST PREDICTIVE SENTIMENT INDICATORS")
print("-" * 40)
top_3 = comparison_df.head(3)
for idx, row in top_3.iterrows():
    print(f"  {row['Sentiment']}: {row['N_Significant']} significant factors "  
          f"({row['Pct_Significant']:.1f}%)")

print("\n" + "="*80)

## Conclusion

This analysis demonstrates that:

1. **Sentiment orthogonalization** successfully isolates the behavioral component from macroeconomic influences

2. **Sentiment-conditional analysis** reveals that factor returns vary significantly across high and low sentiment regimes

3. **Different sentiment indicators** show varying degrees of predictive power for factor returns

### Next Steps:
- Analyze time-variation in sentiment effects
- Test robustness across different sample periods
- Investigate cross-sectional differences (by country, industry, etc.)
- Develop trading strategies based on sentiment signals

### Files Generated:
- `results/figures/sentiment_indicators.png`
- `results/figures/sentiment_orthogonalization_comparison.png`
- `results/figures/high_vs_low_sentiment_returns.png`
- `results/figures/hml_tstatistics.png`
- `results/tables/bw_sentiment_conditional.csv`
- `results/tables/sentiment_comparison.csv`
- `results/checkpoints/orthogonalized_sentiment.csv`